## 2.4 文档嵌入模型 Text Embedding Models
### 2.4.1 嵌入模型概述
Text Embedding Models：文档嵌入模型，提供将文本编码为向量的能力，即 `文档向量化` 。 文档写入 和 用户查询匹配 前都会先执行文档嵌入编码，即向量化。 

常用的嵌入模型:

LangChain中针对向量化模型的封装提供了两种接口，一种针对 句子的向量化embed_query ，一种针
对 文档的向量化(embed_documents) 。

### 2.4.2 嵌入模型的选择 

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from langchain.embeddings import init_embeddings

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")



embedding_model = init_embeddings(
    model="openai:Qwen3-VL-Embedding-2B",
    api_key=os.environ["embedding_model_key"],
    base_url=os.environ["embedding_model_base"],
    #profile=custom_profile
)

index_config = {
    "embed": embedding_model,
    "dims": 512,
    "fields": ["$"]
}

In [5]:
from langchain_openai import OpenAIEmbeddings
embedding_model2 = OpenAIEmbeddings(
    model="jina-embeddings-v4",
    api_key=os.environ["embedding_model_key"],
    base_url=os.environ["embedding_model_base"],
)

### 2.4.3 句子的向量化（embed_query）

In [4]:

text = "你好，很高兴认识你"


embed_docs = embedding_model.embed_query(text)

print(len(embed_docs))

print(embed_docs[:5])

1024
[0.01232147216796875, -0.020965576171875, 0.030303955078125, 0.002002716064453125, 0.0156707763671875]


In [6]:

text = "你好，很高兴认识你"


embed_docs = embedding_model2.embed_query(text)

print(len(embed_docs))

print(embed_docs[:5])

2048
[0.0131248002871871, -0.03309205546975136, 0.004701917991042137, 0.015739163383841515, -0.004711000714451075]


### 2.4.4 文档的向量化（embed_documents）
文档的向量化，接收的参数是字符串数组。

In [7]:

# 待嵌入的文本列表
texts = [
    "Hi there!",
    "Oh, hello!",
    "What's your name?",
    "My friends call me World",
    "Hello World!"
]

# 生成嵌入向量
embeded_docs = embedding_model.embed_documents(texts)

for i in range(len(texts)):
    print(f"{texts[i]}:{embeded_docs[i][:3]}",end="\n\n")

Hi there!:[0.03607177734375, -0.0236663818359375, 0.030609130859375]

Oh, hello!:[0.0147552490234375, -0.0271759033203125, 0.040802001953125]

What's your name?:[0.03277587890625, -0.01226043701171875, 0.00859832763671875]

My friends call me World:[-0.0035839080810546875, -0.038238525390625, 0.0179595947265625]

Hello World!:[-0.01491546630859375, -0.006175994873046875, 0.007595062255859375]



In [8]:

# 待嵌入的文本列表
texts = [
    "Hi there!",
    "Oh, hello!",
    "What's your name?",
    "My friends call me World",
    "Hello World!"
]

# 生成嵌入向量
embeded_docs = embedding_model2.embed_documents(texts)

for i in range(len(texts)):
    print(f"{texts[i]}:{embeded_docs[i][:3]}",end="\n\n")

Hi there!:[0.0020421501249074936, -0.026686308905482292, 0.02462201938033104]

Oh, hello!:[0.0018917950801551342, -0.03210397809743881, -0.00561273330822587]

What's your name?:[-0.006894869264215231, -0.010459912940859795, -0.04930686950683594]

My friends call me World:[-0.01910482533276081, -0.025690464302897453, -0.02106555551290512]

Hello World!:[0.0030771896708756685, -0.029493259266018867, -0.01423666626214981]



csv 文件的向量化

In [ ]:
from langchain_community.document_loaders import CSVLoader


# 情况1：
loader = CSVLoader("load/02-load.csv", encoding="utf-8")
docs = loader.load_and_split()#默认切分使用递归切分的方法

print(len(docs))
for doc in docs:
    print(doc)
# 存放的是每一个chrunk的embedding。

# texts = [doc.page_content for doc in docs]

# embeded_docs = embedding_model2.embed_documents(texts)

# print(len(embeded_docs))

# for i in range(len(texts)):
#     print(f"{texts[i]}:\n{embeded_docs[i][:3]}",end="\n\n")

4
page_content='id: 1
title: Introduction to Python
content: Python is a popular programming language.
author: John Doe' metadata={'source': 'load/02-load.csv', 'row': 0}
page_content='id: 2
title: Data Science Basics
content: Data science involves statistics and machine learning.
author: Jane Smith' metadata={'source': 'load/02-load.csv', 'row': 1}
page_content='id: 3
title: Web Development
content: HTML, CSS and JavaScript are core web technologies.
author: Mike Johnson' metadata={'source': 'load/02-load.csv', 'row': 2}
page_content='id: 4
title: Artificial Intelligence
content: AI is transforming many industries.
author: Sarah Williams' metadata={'source': 'load/02-load.csv', 'row': 3}
